# Sparse autoencoder trên residual stream của GPT-2 small

Neuron và hướng trong residual stream thường **đa nghĩa** (polysemantic): một chiều bật cho nhiều khái niệm không liên quan, vì
model nhét nhiều đặc trưng hơn số chiều (superposition). Sparse autoencoder (SAE) học một từ điển lớn hơn nhiều số chiều, sao cho
mỗi activation được tái dựng bằng **ít** phần tử từ điển. Bạn tự viết SAE (bản ReLU + L1 và bản TopK), kiểm trên dữ liệu đồ chơi
có đáp án, rồi train trên residual stream tầng giữa của GPT-2 small, đo bằng ba trục: tái dựng (FVU), độ thưa (L0, latent chết)
và độ trung thành với model (cross-entropy khi ghép SAE vào), cuối cùng đọc vài latent bằng ví dụ kích hoạt mạnh nhất.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm/lớp. Điền vào chỗ `# TODO`.
- Cell `explore` tải model, thu activation, train; test bỏ qua chúng.
- GPU 4GB (kernel `ai-gpu`): 1 triệu token, từ điển 6 144 latent. Không có GPU: notebook tự giảm còn 200 nghìn token và 3 072
  latent, kết luận vẫn kiểm được, số tuyệt đối kém hơn.
- Chấm: `uv run pytest tracks/ai-engineer/modules/08-interpretability/exercises/intermediate/03-sparse-autoencoder -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải IWSLT15 nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
IWSLT = ROOT / "data/raw/iwslt15-en-vi"
if not (IWSLT.is_dir() and any(IWSLT.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("iwslt15-en-vi")
print(sorted(p.name for p in IWSLT.iterdir()))

In [ ]:
import contextlib

import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Sparse autoencoder

$$
f(x) = \sigma\big((x - b_{dec})W_{enc} + b_{enc}\big), \qquad \hat x = f(x)W_{dec} + b_{dec}
$$

$$\sigma$$ là ReLU (khi đó độ thưa đến từ phạt L1 trong loss) hoặc TopK (giữ đúng $$k$$ latent lớn nhất, không cần phạt). Hàng $$i$$ của
$$W_{dec}$$ là "hướng" của latent $$i$$ trong residual stream. Phạt L1 nhân với $$\lVert W_{dec,i}\rVert$$: nếu không, SAE ăn gian bằng cách
thu nhỏ $$f$$ và phóng to hàng decoder tương ứng.

In [ ]:
class SparseAutoencoder(nn.Module):
    """SAE một tầng ẩn trên activation shape (..., d_in), d_sae latent.

    - Khởi tạo: `W_dec` (d_sae, d_in) rút `torch.randn` bằng `torch.Generator().manual_seed(seed)` rồi chuẩn hoá mỗi hàng về độ
      dài 1; `W_enc` (d_in, d_sae) là bản sao của `W_dec.T`; `b_enc` (d_sae,), `b_dec` (d_in,) bằng 0. Bốn `nn.Parameter`.
    - `encode(x)`: pre = (x - b_dec) @ W_enc + b_enc. `k is None`: ReLU(pre). Ngược lại (TopK): giữ k giá trị pre lớn nhất mỗi
      hàng (qua ReLU), còn lại 0.
    - `decode(f)` = f @ W_dec + b_dec. `forward(x)` trả (x_hat, f).
    - `loss(x)` trả (loss, info): mse = trung bình theo hàng của tổng bình phương lỗi theo chiều cuối; l1 = trung bình theo
      hàng của tổng f_i * ||W_dec[i]||; loss = mse + l1_coeff * l1; info là dict float {"mse", "l1", "l0"} (l0 = số latent
      khác 0 trung bình mỗi hàng).
    - `normalize_decoder_()`: chuẩn hoá tại chỗ mỗi hàng của W_dec về độ dài 1 (không gradient).
    """

    def __init__(self, d_in: int, d_sae: int, k: int | None = None, l1_coeff: float = 0.0, seed: int = 0):
        super().__init__()
        # TODO
        raise NotImplementedError

    def encode(self, x):
        # TODO
        raise NotImplementedError

    def decode(self, f):
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

    def loss(self, x):
        # TODO
        raise NotImplementedError

    def normalize_decoder_(self):
        # TODO
        raise NotImplementedError

## 2. Train và đánh giá

- `activation_scale`: residual stream của GPT-2 có norm cỡ vài chục và khác nhau giữa các tầng. Nhân một hằng số để
  $$\mathbb{E}\lVert x\rVert^2 = d$$ giúp cùng một learning rate và cùng hệ số L1 dùng được cho mọi tầng, mọi model.
- `train_sae`: Adam, batch ngẫu nhiên, chuẩn hoá decoder sau mỗi bước, lr giảm về 0 ở 20% bước cuối.
- `sae_metrics`: FVU (fraction of variance unexplained, $$1 - R^2$$), L0, tỉ lệ latent chết.

In [ ]:
def activation_scale(acts):
    """Hằng số c để trung bình ||c * x||^2 bằng d_in (x là hàng của `acts`, shape (N, d_in)): c = sqrt(d_in / mean ||x||^2).
    Trả float. Tính bằng float32 kể cả khi `acts` là float16."""
    # TODO
    raise NotImplementedError

In [ ]:
def train_sae(sae, acts, steps, batch_size=4096, lr=4e-4, scale=1.0, seed=0, device="cpu"):
    """Train `sae` bằng Adam(lr) trong `steps` bước. Mỗi bước: chỉ số batch rút bằng
    `torch.randint(len(acts), (batch_size,), generator=g)` với `g = torch.Generator().manual_seed(seed)`;
    x = acts[idx] chuyển sang `device`, float32, nhân `scale`; loss của `sae.loss`; sau `opt.step()` gọi
    `sae.normalize_decoder_()`. Learning rate giữ `lr` rồi giảm tuyến tính về 0 trong 20% số bước cuối.
    Trả list loss (float) từng bước.
    """
    # TODO
    raise NotImplementedError

In [ ]:
@torch.no_grad()
def sae_metrics(sae, acts, scale=1.0, batch_size=4096, device="cpu"):
    """Đánh giá trên `acts` (N, d_in), mỗi hàng nhân `scale` trước khi đưa vào SAE. Trả dict float:
    - `fvu`: tổng ||x - x_hat||^2 chia tổng ||x - mean(x)||^2 (mean theo mọi hàng của `acts` đã nhân scale);
    - `l0`: số latent khác 0 trung bình mỗi hàng;
    - `dead_frac`: tỉ lệ latent **không bao giờ** khác 0 trên toàn bộ `acts`.
    """
    # TODO
    raise NotImplementedError

## 3. Activation thật và độ trung thành với model

- `collect_resid_pre`: thu residual stream vào một tầng của GPT-2, bỏ vị trí 0. Vị trí 0 của GPT-2 có norm lớn gấp hàng chục lần
  các vị trí khác ("attention sink"); để nó trong dữ liệu train là để vài hàng ngoại lai chi phối MSE.
- `sae_splice`: ghép SAE vào model (thay stream bằng bản tái dựng) để đo model còn dự đoán tốt tới đâu.
- `ce_loss_recovered`: đặt CE khi ghép SAE giữa CE của model gốc và CE khi ablation tầng đó.

In [ ]:
@torch.no_grad()
def collect_resid_pre(model, contexts, layer, batch_size=32, device="cpu"):
    """Residual stream đi vào Block `layer` (stream trước `ln1`) của mọi ngữ cảnh trong `contexts` (N, T), **bỏ vị trí 0**
    (token đầu `<|endoftext|>`). Trả tensor float16 trên CPU shape (N * (T - 1), C), xếp theo ngữ cảnh rồi theo vị trí.
    Chạy theo batch, gỡ hook khi xong.
    """
    # TODO
    raise NotImplementedError

In [ ]:
@contextlib.contextmanager
def sae_splice(model, layer, sae, scale):
    """Trong khối `with`, residual stream đi vào Block `layer` ở mọi vị trí trừ vị trí 0 được thay bằng bản tái dựng của SAE:
    x -> sae(x * scale)[0] / scale. Vị trí 0 giữ nguyên. Gỡ hook khi ra khỏi khối, kể cả khi có exception.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def ce_loss_recovered(ce_clean, ce_sae, ce_ablated):
    """Tỉ lệ loss được "khôi phục" khi thay activation bằng bản tái dựng của SAE: 1 khi bằng model gốc, 0 khi tệ như ablation."""
    # TODO
    raise NotImplementedError

## 4. Thí nghiệm

Các cell dưới không được chấm. Thứ tự: kiểm SAE trên dữ liệu đồ chơi có đáp án, rồi GPT-2.

In [ ]:
# explore: dữ liệu đồ chơi: 30 đặc trưng thưa trong 10 chiều (superposition); SAE TopK có tìm lại được 30 hướng không?
import time

torch.set_num_threads(4)
g = torch.Generator().manual_seed(0)
true_dirs = torch.randn(30, 10, generator=g)
true_dirs /= true_dirs.norm(dim=1, keepdim=True)
active = (torch.rand(20000, 30, generator=g) < 0.05).float() * (0.5 + torch.rand(20000, 30, generator=g))
toy = active @ true_dirs
for name, kw in [("TopK k=3", dict(k=3)), ("ReLU + L1 0.05", dict(l1_coeff=0.05))]:
    sae = SparseAutoencoder(10, 60, seed=0, **kw)
    train_sae(sae, toy, steps=1500, batch_size=256, lr=3e-3)
    cos = (true_dirs @ sae.W_dec.detach().T).max(dim=1).values
    m = sae_metrics(sae, toy)
    print(f"{name:15s} FVU {m['fvu']:.3f}  L0 {m['l0']:.1f}  cosine tốt nhất với 30 hướng thật: trung bình {cos.mean():.3f}, nhỏ nhất {cos.min():.3f}")

In [ ]:
# explore: nạp GPT-2 small vào lớp GPT (dùng gpt2_into_gpt_mini của bài Intermediate 01, chưa xong thì lời giải)
import importlib.util

import json
from transformers import AutoTokenizer, GPT2LMHeadModel

M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"
M08 = ROOT / "tracks/ai-engineer/modules/08-interpretability"
for path in [M05 / "exercises/basic/01-gpt-mini/starter/gpt.py", M05 / "solutions/basic/01-gpt-mini/gpt.py"]:
    spec = importlib.util.spec_from_file_location("gpt_mini", path)
    gpt_mini = importlib.util.module_from_spec(spec)
    sys.modules["gpt_mini"] = gpt_mini
    spec.loader.exec_module(gpt_mini)
    try:
        gpt_mini.GPT(gpt_mini.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long))
        break
    except NotImplementedError:
        continue


def notebook_namespace(path):
    """Chạy các cell code không gắn tag explore của một notebook, trả namespace (giống cách test nạp notebook)."""
    ns = {}
    for cell in json.loads(Path(path).read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code" and "explore" not in cell["metadata"].get("tags", []):
            exec("".join(cell["source"]), ns)
    return ns


def load_gpt2(gpt_mini):
    """GPT-2 small trong lớp GPT bằng gpt2_into_gpt_mini của bạn (bài Intermediate 01); lệch logits so với Hugging Face quá 1e-3
    hoặc lỗi bất kỳ thì dùng lời giải và in cảnh báo."""
    I01 = ROOT / "tracks/ai-engineer/modules/08-interpretability/{}/intermediate/01-gpt2-heads-lens-probe/{}inside_gpt2.ipynb"
    hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
    probe_ids = torch.tensor([[50256, 464, 412, 733, 417, 8765, 318, 5140, 287, 262, 1748, 286]])
    with torch.no_grad():
        ref = hf(probe_ids).logits
    for d, sub in [("exercises", "starter/"), ("solutions", "")]:
        try:
            mdl = notebook_namespace(str(I01).format(d, sub))["gpt2_into_gpt_mini"](gpt_mini, hf)
            with torch.no_grad():
                diff = (mdl(probe_ids)[0] - ref).abs().max().item()
            if diff <= 1e-3:
                print(f"gpt2_into_gpt_mini từ {d} (bài Intermediate 01), lệch logits {diff:.1e}")
                return mdl.eval()
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lệch logits {diff:.1e} > 1e-3, chuyển sang lời giải")
        except NotImplementedError:
            continue
        except Exception as e:
            print(f"cảnh báo: gpt2_into_gpt_mini ở {d} lỗi ({type(e).__name__}: {e}), chuyển sang lời giải")
    raise RuntimeError("không nạp được GPT-2")


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = load_gpt2(gpt_mini).to(DEVICE).eval()
BOS = tok.bos_token_id
print("GPT-2 small trên", DEVICE)

In [ ]:
# explore: thu residual stream vào Block 6 trên văn bản TED (IWSLT15 train.en), giữ tst2012 + tst2013 làm tập đánh giá
import html
import re


def detok(s):
    s = html.unescape(s)
    s = re.sub(r" ([.,!?;:%)\]])", r"\1", s)
    s = re.sub(r"([(\[$]) ", r"\1", s)
    s = re.sub(r" (n't|'s|'re|'ve|'m|'ll|'d)\b", r"\1", s)
    return re.sub(r'" (.*?) "', r'"\1"', s)


def contexts(lines, n_tokens, ctx=128):
    ids = tok.encode(" ".join(detok(s) for s in lines))
    n = min(len(ids) // (ctx - 1), -(-n_tokens // (ctx - 1)))
    x = torch.tensor(ids[: n * (ctx - 1)]).view(n, ctx - 1)
    return torch.cat([torch.full((n, 1), BOS), x], dim=1)


LAYER = 6
N_TOKENS, D_SAE = (1_000_000, 8 * 768) if DEVICE == "cuda" else (200_000, 4 * 768)
lines = (IWSLT / "train.en").read_text(encoding="utf-8").splitlines()[:120_000]
eval_lines = [l for s in ["tst2012", "tst2013"] for l in (IWSLT / f"{s}.en").read_text(encoding="utf-8").splitlines()]
train_ctx, eval_ctx = contexts(lines, N_TOKENS), contexts(eval_lines, 10**9)
start = time.perf_counter()
acts = collect_resid_pre(model, train_ctx, LAYER, batch_size=16, device=DEVICE)
eval_acts = collect_resid_pre(model, eval_ctx, LAYER, batch_size=16, device=DEVICE)
print(f"train {tuple(acts.shape)}, eval {tuple(eval_acts.shape)}, {time.perf_counter() - start:.0f} s")
with torch.no_grad():
    store = {}
    h = model.blocks[LAYER].register_forward_pre_hook(lambda m, a: store.__setitem__("x", a[0]))
    model(eval_ctx[:16].to(DEVICE))
    h.remove()
norms = store["x"].norm(dim=-1)
print(f"norm stream tầng {LAYER}: vị trí 0 trung bình {norms[:, 0].mean():.0f}, các vị trí khác {norms[:, 1:].mean():.1f}")
SCALE = activation_scale(acts)
print(f"scale = {SCALE:.4f}")

In [ ]:
# explore: train SAE TopK (k = 32) và hai SAE ReLU + L1, 4 lượt qua dữ liệu; đo FVU, L0, latent chết trên tập đánh giá
steps = 4 * len(acts) // 4096
saes = {}
for name, kw in [("TopK k=32", dict(k=32)), ("L1 2.0", dict(l1_coeff=2.0)), ("L1 5.0", dict(l1_coeff=5.0))]:
    start = time.perf_counter()
    sae = SparseAutoencoder(768, D_SAE, seed=0, **kw)
    hist = train_sae(sae, acts, steps, scale=SCALE, device=DEVICE)
    m = sae_metrics(sae, eval_acts, scale=SCALE, device=DEVICE)
    saes[name] = sae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{name:10s} {steps} bước, {time.perf_counter() - start:.0f} s | FVU {m['fvu']:.3f}  L0 {m['l0']:.1f}  chết {m['dead_frac']:.3f}")

In [ ]:
# explore: ghép SAE vào model: cross-entropy trên tập đánh giá, so với model gốc, zero ablation và mean ablation tầng đó
@torch.no_grad()
def eval_ce(ctx, patch=None):
    total, count = 0.0, 0
    h = model.blocks[LAYER].register_forward_pre_hook(patch) if patch else None
    try:
        for i in range(0, len(ctx), 8):
            x = ctx[i:i + 8].to(DEVICE)
            logits = model(x)[0]
            total += F.cross_entropy(logits[:, 1:-1].transpose(1, 2), x[:, 2:], reduction="sum").item()
            count += x[:, 2:].numel()
    finally:
        if h:
            h.remove()
    return total / count


def replace_with(value):
    def hook(module, args):
        x = args[0].clone()
        x[:, 1:] = value
        return (x,)
    return hook


mean_act = acts.float().mean(dim=0).to(DEVICE)
ce = {"gốc": eval_ce(eval_ctx), "zero ablation": eval_ce(eval_ctx, replace_with(0.0)), "mean ablation": eval_ce(eval_ctx, replace_with(mean_act))}
print("  ".join(f"{k} {v:.3f}" for k, v in ce.items()))
for name, sae in saes.items():
    with sae_splice(model, LAYER, sae, SCALE):
        ce_sae = eval_ce(eval_ctx)
    print(f"{name:10s} CE {ce_sae:.3f} | recovered (so với zero) {ce_loss_recovered(ce['gốc'], ce_sae, ce['zero ablation']):.3f}"
          f" | (so với mean) {ce_loss_recovered(ce['gốc'], ce_sae, ce['mean ablation']):.3f}")

In [ ]:
# explore: đọc latent của SAE TopK: mật độ kích hoạt, và ví dụ kích hoạt mạnh nhất của vài latent (token kích hoạt trong [ ])
sae = saes["TopK k=32"]
with torch.no_grad():
    f_eval = torch.cat([sae.encode(eval_acts[i:i + 4096].to(DEVICE).float() * SCALE).cpu() for i in range(0, len(eval_acts), 4096)])
density = (f_eval > 0).float().mean(0)
alive = (density > 0).nonzero().flatten()
print(f"latent sống {len(alive)}/{len(density)}; mật độ trung vị (latent sống) {density[alive].median():.2e}")
tokens = eval_ctx[:, 1:].reshape(-1)
T1 = eval_ctx.shape[1] - 1


def show(latent, k=6, width=8):
    vals, idx = f_eval[:, latent].topk(k)
    print(f"--- latent {latent}: mật độ {density[latent]:.2e}")
    for v, i in zip(vals.tolist(), idx.tolist()):
        row, pos = divmod(i, T1)
        t = pos + 1
        left = tok.decode(eval_ctx[row, max(1, t - width):t])
        here = tok.decode(eval_ctx[row, t:t + 1])
        nxt = tok.decode(eval_ctx[row, t + 1:t + 3])
        print(f"  {v:6.2f} | ...{left}[{here}]{nxt}")


g = torch.Generator().manual_seed(0)
candidates = alive[(density[alive] > 1e-4) & (density[alive] < 1e-2)]
for latent in candidates[torch.randperm(len(candidates), generator=g)[:6]].tolist():
    show(latent)

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.hist(torch.log10(density[alive]).numpy(), bins=50)
plt.xlabel("log10 mật độ kích hoạt"); plt.ylabel("số latent"); plt.title("Mật độ latent (SAE TopK, tập đánh giá)")
plt.tight_layout(); plt.show()

## 5. Đọc kết quả

Trả lời cho chính mình (rồi làm phần **trắc nghiệm** trong app):

1. Dữ liệu đồ chơi: SAE TopK có tìm lại các hướng thật không? ReLU + L1 thì sao, và L0 của nó so với số đặc trưng thật đang bật
   mỗi mẫu?
2. Ba SAE trên GPT-2: vẽ chúng trên mặt phẳng (L0, FVU). Bản nào tốt hơn ở cùng ngân sách train, và vì sao so FVU mà không so L0 là
   vô nghĩa?
3. CE recovered: con số đổi bao nhiêu khi đổi baseline từ zero ablation sang mean ablation? Báo cáo SAE của người khác thì bạn hỏi
   lại câu gì?
4. Latent: đọc 6 latent ngẫu nhiên. Bao nhiêu cái bạn đặt được tên? Ví dụ kích hoạt mạnh nhất cho thấy gì và **không** cho thấy gì
   về ý nghĩa của latent (gợi ý: các mức kích hoạt yếu, và latent có gây ra hành vi nào không)?